<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/Analise_de_Raking_Funil_Robustez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import pandas as pd
# 1. Carregamento da base de dados
# Corrigido o separador de ';' para ',' pois a base utiliza vírgulas.
caminho_arquivo = '/content/drive/MyDrive/dataset_mestre_observatorio.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=',', encoding='utf-8')

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Lendo o arquivo original...
Base carregada com sucesso! O arquivo inteiro tem 5570 linhas e 15 colunas.


### 🔍 Diagnóstico das Variáveis de Elegibilidade

Antes de aplicarmos os filtros finais, é importante inspecionar a qualidade, a presença de dados nulos e a formatação das principais colunas no dataset mestre. Isso garante que nenhum dado relevante seja excluído acidentalmente devido a diferenças de digitação ou representação de valores lógicos.

In [14]:
# 1. Quantas linhas sua tabela tem antes dos filtros?
print("Total de municípios na tabela mestre:", len(df))

# 2. Como estão gravadas as colunas de filtro?
print("\nValores em 'participou_sinisa':", df['participou_sinisa'].unique())
print("Valores em 'mapeamento_area_risco':", df['mapeamento_area_risco'].unique())
print("Valores em 'capag':", df['capag'].unique())

# Diagnóstico extra: contagem de valores nulos ou filtros muito restritivos
print("\nMunicípios com participou_sinisa == 'Sim':", len(df[df['participou_sinisa'] == 'Sim']))
print("Municípios com mapeamento_area_risco == 'Sim':", len(df[df['mapeamento_area_risco'] == 'Sim']))
print("Municípios com qtd_domicilios_risco > 0:", len(df[df['qtd_domicilios_risco'] > 0]))

Total de municípios na tabela mestre: 5570

Valores em 'participou_sinisa': [ True False]
Valores em 'mapeamento_area_risco': ['Sim' 'Não' nan]
Valores em 'capag': ['C' 'B+' 'A' 'A+' 'B' 'n.d.' 'n.e.' 'D' nan]

Municípios com participou_sinisa == 'Sim': 0
Municípios com mapeamento_area_risco == 'Sim': 1460
Municípios com qtd_domicilios_risco > 0: 2238


### 📊 Pipeline de Elegibilidade e Teste de Robustez dos Cenários

Este bloco de código consolida duas etapas fundamentais do nosso estudo:

1. **Tratamento e Aplicação de Filtros (Elegibilidade):** O algoritmo realiza a limpeza e padronização das bases, tratando valores nulos e convertendo colunas de texto e booleanos para aplicar de forma rigorosa os quatro critérios de elegibilidade (participação no **SINISA**, existência de **Mapeamento de Risco**, exclusão de **CAPAG** 'A' e 'B', e presença de **Domicílios em Risco**). Ao final, o diagnóstico exibe quantos municípios restaram em cada etapa até chegar aos **641 elegíveis**.
2. **Simulação de Pesos (Análise de Sensibilidade):** Com as cidades elegíveis definidas, o código simula de forma automatizada quatro cenários ponderando a escala absoluta (quantidade de domicílios em risco) e a vulnerabilidade local (percentual de domicílios em risco). A tabela de robustez gerada calcula o trade-off de moradias atendidas no Top 20 e o nível de sobreposição de municípios em relação ao Cenário Base.

In [15]:
import pandas as pd
import numpy as np

# 1. Criar uma cópia de trabalho
df_test = df.copy()

# 2. Filtro 1: SINISA (Reconhece o valor lógico True)
filtro_sinisa = (df_test['participou_sinisa'] == True) | (df_test['participou_sinisa'].astype(str).str.strip().str.upper().isin(['TRUE', 'SIM', '1']))

# 3. Filtro 2: Mapeamento de Risco (Reconhece 'Sim')
filtro_mapeamento = df_test['mapeamento_area_risco'].fillna('').astype(str).str.strip().str.upper().isin(['SIM', 'S', '1', 'TRUE'])

# 4. Filtro 3: CAPAG (Exclui notas A, A+, B e B+ conforme Decisão 1B)
capag_limpo = df_test['capag'].fillna('').astype(str).str.strip().str.upper()
filtro_capag = ~capag_limpo.isin(['A', 'A+', 'B', 'B+'])

# 5. Filtro 4: Moradias em Risco (> 0)
df_test['qtd_domicilios_risco'] = pd.to_numeric(df_test['qtd_domicilios_risco'], errors='coerce').fillna(0)
df_test['pct_domicilios_risco'] = pd.to_numeric(df_test['pct_domicilios_risco'], errors='coerce').fillna(0)
filtro_risco = df_test['qtd_domicilios_risco'] > 0

# 6. Filtrar a lista final de elegíveis
df_elegiveis = df_test[filtro_sinisa & filtro_mapeamento & filtro_capag & filtro_risco].copy()

print("=== DIAGNÓSTICO DOS FILTROS ===")
print(f"Total na tabela mestre: {len(df_test)}")
print(f"1. Participou do SINISA (True): {len(df_test[filtro_sinisa])}")
print(f"2. Mapeamento de Risco (Sim): {len(df_test[filtro_mapeamento])}")
print(f"3. CAPAG Elegível (sem A, A+, B, B+): {len(df_test[filtro_capag])}")
print(f"4. Domicílios em Risco (>0): {len(df_test[filtro_risco])}")
print(f"--> ELEGÍVEIS FINAIS: {len(df_elegiveis)}")

# 7. Rodar o Teste de Robustez (Decisão 3)
if len(df_elegiveis) > 0:
    # Transforma em percentil (régua de 0.0 a 1.0)
    df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
    df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

    cenarios = [
        (1.0, 0.0, "100% Qtd / 0% Pct (Cenário Base)"),
        (0.7, 0.3, "70% Qtd / 30% Pct"),
        (0.5, 0.5, "50% Qtd / 50% Pct"),
        (0.0, 1.0, "0% Qtd / 100% Pct")
    ]

    top20_base_ids = set(df_elegiveis.nlargest(20, 'rank_qtd_pctile')['Cod_IBGE_7'])

    resultados = []
    for peso_qtd, peso_pct, nome_cenario in cenarios:
        df_elegiveis['nota_final'] = (peso_qtd * df_elegiveis['rank_qtd_pctile']) + (peso_pct * df_elegiveis['rank_pct_pctile'])
        top20_cenario = df_elegiveis.nlargest(20, 'nota_final')
        top20_cenario_ids = set(top20_cenario['Cod_IBGE_7'])

        permanecem = len(top20_cenario_ids.intersection(top20_base_ids))
        total_moradias = top20_cenario['qtd_domicilios_risco'].sum()

        resultados.append({
            'Cenário de Pesos': nome_cenario,
            'Permanecem do Top 20 Base': f"{permanecem} de 20",
            'Moradias Cobertas no Top 20': f"{total_moradias:,.0f}"
        })

    df_robustez = pd.DataFrame(resultados)
    print("\n=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===")
    print(df_robustez.to_string(index=False))

=== DIAGNÓSTICO DOS FILTROS ===
Total na tabela mestre: 5570
1. Participou do SINISA (True): 4845
2. Mapeamento de Risco (Sim): 1460
3. CAPAG Elegível (sem A, A+, B, B+): 3026
4. Domicílios em Risco (>0): 2238
--> ELEGÍVEIS FINAIS: 641

=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===
                Cenário de Pesos Permanecem do Top 20 Base Moradias Cobertas no Top 20
100% Qtd / 0% Pct (Cenário Base)                  20 de 20                     698,440
               70% Qtd / 30% Pct                  10 de 20                     598,353
               50% Qtd / 50% Pct                   7 de 20                     551,362
               0% Qtd / 100% Pct                   3 de 20                     165,342


### ⚙️ Detalhamento do Pipeline de Elegibilidade e Robustez

Este bloco de código executa as etapas lógicas cruciais para a consolidação das análises. Abaixo está o detalhamento de como cada parte do script opera:

---

#### 1. Criação do DataFrame de Trabalho
* O código cria uma cópia isolada (`df_test`) do DataFrame principal. Isso evita qualquer efeito colateral indesejado nas variáveis globais do seu ambiente.

#### 2. Normalização e Limpeza dos Filtros (Etapas 2 a 5)
* **SINISA (`filtro_sinisa`):** Garante a inclusão dos municípios participantes convertendo valores lógicos e strings de texto (`'Sim'`, `'True'`, `'1'`) de forma uniforme para maiúsculas e sem espaços adicionais.
* **Mapeamento de Risco (`filtro_mapeamento`):** Identifica e filtra municípios com planos e diagnósticos ativos, preenchendo eventuais nulos com vazios para evitar erros de comparação lógica.
* **CAPAG (`filtro_capag`):** Exclui de forma robusta e dinâmica as classificações de crédito mais favoráveis (`A`, `A+`, `B` e `B+`), aplicando o operador de negação (`~`).
* **Domicílios em Risco (`filtro_risco`):** Força a conversão das métricas absolutas e proporcionais para o tipo numérico, tratando dados inválidos como zero e garantindo que apenas cidades com vulnerabilidade real mapeada sejam mantidas.

#### 3. Diagnóstico Iterativo dos Filtros (Etapa 6)
* Imprime na tela um balanço passo a passo do funil de elegibilidade, permitindo auditar o impacto individual de cada um dos critérios definidos.

#### 4. Simulação de Sensibilidade por Percentil (Etapa 7)
* **Régua de Percentis:** Transforma as colunas brutas em percentis (`rank(pct=True)`), colocando os diferentes eixos (absoluto vs. proporcional) na mesma escala de comparação (de 0.0 a 1.0).
* **Notas Ponderadas:** Calcula e simula as notas finais dos municípios nos quatro cenários de peso definidos.
* **Tabela de Robustez:** Apresenta a contagem de moradias cobertas no Top 20 de cada cenário e compara o grau de permanência das cidades de alta prioridade frente ao modelo de referência absoluto.

In [ ]:
import pandas as pd
import numpy as np

# 1. Criar uma cópia de trabalho
df_test = df.copy()

# 2. Filtro 1: SINISA (Reconhece o valor lógico True)
filtro_sinisa = (df_test['participou_sinisa'] == True) | (df_test['participou_sinisa'].astype(str).str.strip().str.upper().isin(['TRUE', 'SIM', '1']))

# 3. Filtro 2: Mapeamento de Risco (Reconhece 'Sim')
filtro_mapeamento = df_test['mapeamento_area_risco'].fillna('').astype(str).str.strip().str.upper().isin(['SIM', 'S', '1', 'TRUE'])

# 4. Filtro 3: CAPAG (Exclui notas A, A+, B e B+ conforme Decisão 1B)
capag_limpo = df_test['capag'].fillna('').astype(str).str.strip().str.upper()
filtro_capag = ~capag_limpo.isin(['A', 'A+', 'B', 'B+'])

# 5. Filtro 4: Moradias em Risco (> 0)
df_test['qtd_domicilios_risco'] = pd.to_numeric(df_test['qtd_domicilios_risco'], errors='coerce').fillna(0)
df_test['pct_domicilios_risco'] = pd.to_numeric(df_test['pct_domicilios_risco'], errors='coerce').fillna(0)
filtro_risco = df_test['qtd_domicilios_risco'] > 0

# 6. Filtrar a lista final de elegíveis
df_elegiveis = df_test[filtro_sinisa & filtro_mapeamento & filtro_capag & filtro_risco].copy()

print("=== DIAGNÓSTICO DOS FILTROS ===")
print(f"Total na tabela mestre: {len(df_test)}")
print(f"1. Participou do SINISA (True): {len(df_test[filtro_sinisa])}")
print(f"2. Mapeamento de Risco (Sim): {len(df_test[filtro_mapeamento])}")
print(f"3. CAPAG Elegível (sem A, A+, B, B+): {len(df_test[filtro_capag])}")
print(f"4. Domicílios em Risco (>0): {len(df_test[filtro_risco])}")
print(f"--> ELEGÍVEIS FINAIS: {len(df_elegiveis)}")

# 7. Rodar o Teste de Robustez (Decisão 3)
if len(df_elegiveis) > 0:
    # Transforma em percentil (régua de 0.0 a 1.0)
    df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
    df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

    cenarios = [
        (1.0, 0.0, "100% Qtd / 0% Pct (Cenário Base)"),
        (0.7, 0.3, "70% Qtd / 30% Pct"),
        (0.5, 0.5, "50% Qtd / 50% Pct"),
        (0.0, 1.0, "0% Qtd / 100% Pct")
    ]

    top20_base_ids = set(df_elegiveis.nlargest(20, 'rank_qtd_pctile')['Cod_IBGE_7'])

    resultados = []
    for peso_qtd, peso_pct, nome_cenario in cenarios:
        df_elegiveis['nota_final'] = (peso_qtd * df_elegiveis['rank_qtd_pctile']) + (peso_pct * df_elegiveis['rank_pct_pctile'])
        top20_cenario = df_elegiveis.nlargest(20, 'nota_final')
        top20_cenario_ids = set(top20_cenario['Cod_IBGE_7'])

        permanecem = len(top20_cenario_ids.intersection(top20_base_ids))
        total_moradias = top20_cenario['qtd_domicilios_risco'].sum()

        resultados.append({
            'Cenário de Pesos': nome_cenario,
            'Permanecem do Top 20 Base': f"{permanecem} de 20",
            'Moradias Cobertas no Top 20': f"{total_moradias:,.0f}"
        })

    df_robustez = pd.DataFrame(resultados)
    print("\n=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===")
    print(df_robustez.to_string(index=False))

=== DIAGNÓSTICO DOS FILTROS ===
Total na tabela mestre: 5570
1. Participou do SINISA (True): 4845
2. Mapeamento de Risco (Sim): 1460
3. CAPAG Elegível (sem A, A+, B, B+): 3026
4. Domicílios em Risco (>0): 2238
--> ELEGÍVEIS FINAIS: 641

=== TABELA DE ROBUSTEZ DA LISTA SELECIONADA ===
                Cenário de Pesos Permanecem do Top 20 Base Moradias Cobertas no Top 20
100% Qtd / 0% Pct (Cenário Base)                  20 de 20                     698,440
               70% Qtd / 30% Pct                  10 de 20                     598,353
               50% Qtd / 50% Pct                   7 de 20                     551,362
               0% Qtd / 100% Pct                   3 de 20                     165,342


In [16]:
import pandas as pd

# 1. Garantir que as colunas numéricas do Atlas sejam números
cols_atlas = ['total_desastres', 'soma_danos_materiais', 'soma_afetados']
for col in cols_atlas:
    df_elegiveis[col] = pd.to_numeric(df_elegiveis[col], errors='coerce').fillna(0)

# 2. Separar os 20 Campeões (Cenário 70% Qtd / 30% Pct) dos demais elegíveis
df_elegiveis['nota_final'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
top20 = df_elegiveis.nlargest(20, 'nota_final')
demais_elegiveis = df_elegiveis[~df_elegiveis['Cod_IBGE_7'].isin(top20['Cod_IBGE_7'])]

# 3. Montar o Resumo do "Custo de Não Agir"
tabela_custo_nao_agir = pd.DataFrame([
    {
        'Grupo': '20 Selecionados (Top 20)',
        'Qtd Cidades': len(top20),
        'Moradias em Risco (SINISA)': f"{top20['qtd_domicilios_risco'].sum():,.0f}",
        'Total de Desastres (2021-2025)': f"{top20['total_desastres'].sum():,.0f}",
        'Danos Materiais Totais (R$)': f"R$ {top20['soma_danos_materials'].sum() if 'soma_danos_materials' in top20.columns else top20['soma_danos_materiais'].sum():,.2f}",
        'Pessoas Afetadas (Danos Humanos)': f"{top20['soma_afetados'].sum():,.0f}"
    },
    {
        'Grupo': 'Demais Elegíveis (Fora do Top 20)',
        'Qtd Cidades': len(demais_elegiveis),
        'Moradias em Risco (SINISA)': f"{demais_elegiveis['qtd_domicilios_risco'].sum():,.0f}",
        'Total de Desastres (2021-2025)': f"{demais_elegiveis['total_desastres'].sum():,.0f}",
        'Danos Materiais Totais (R$)': f"R$ {demais_elegiveis['soma_danos_materials'].sum() if 'soma_danos_materials' in demais_elegiveis.columns else demais_elegiveis['soma_danos_materiais'].sum():,.2f}",
        'Pessoas Afetadas (Danos Humanos)': f"{demais_elegiveis['soma_afetados'].sum():,.0f}"
    }
])

print("=== TABELA: CUSTO DE NÃO AGIR ===")
print(tabela_custo_nao_agir.to_string(index=False))

=== TABELA: CUSTO DE NÃO AGIR ===
                            Grupo  Qtd Cidades Moradias em Risco (SINISA) Total de Desastres (2021-2025) Danos Materiais Totais (R$) Pessoas Afetadas (Danos Humanos)
         20 Selecionados (Top 20)           20                    598,353                            164           R$ 628,128,636.39                          275,775
Demais Elegíveis (Fora do Top 20)          621                    525,525                          1,649         R$ 9,202,919,613.21                          792,960


### 💾 Exportação e Consolidação Multicritério do Universo de Elegíveis

Este bloco de código é responsável por estruturar o banco de dados final com o histórico completo de posições de todos os **641 municípios elegíveis** sob diferentes visões metodológicas. Veja o detalhamento do processo:

---

#### 1. Cálculo de Notas Simultâneas (Etapas 1 e 2)
* O script recalcula as notas individuais dos 4 cenários de pesos simulados (Base 100/0, Recomendado 70/30, Equilibrado 50/50 e Proporcional 0/100) utilizando as colunas normalizadas por percentil.

#### 2. Geração de Rankings Dinâmicos (Etapa 3)
* Utilizando a função `.rank(ascending=False, method='min')`, o script atribui a posição exata de cada município (do 1º ao 641º lugar) para cada um dos quatro modelos de pesos. Isso permite visualizar como um município sobe ou desce no ranking à medida que o foco político migra da escala absoluta de moradias para a escala de vulnerabilidade proporcional.

#### 3. Sinalização de Alta Prioridade (Etapa 4)
* Cria uma coluna booleana (`is_top20_recomendado`) que marca com `True` apenas as 20 primeiras cidades do cenário recomendado (70/30), servindo como filtro rápido para análises e integrações em ferramentas de mapas ou relatórios.

#### 4. Geração dos Arquivos de Saída (Etapas 5 e 6)
* O banco é ordenado de forma lógica a partir do ranking recomendado (70/30).
* Exporta o conjunto completo de dados e rankings para os formatos **CSV** (otimizado para sistemas de Geoprocessamento e bancos de dados) e **Excel** (formatado para leitura direta por gestores públicos), engatando em seguida o download automático no seu navegador.

In [17]:
import pandas as pd
from google.colab import files

# 1. Garantir o cálculo da nota combinada (70/30) e selecionar os Top 20
df_elegiveis['nota_final'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
top20 = df_elegiveis.nlargest(20, 'nota_final').copy()

# 2. Selecionar as colunas principais para relatório e geoprocessamento
colunas_desejadas = [
    'Cod_IBGE_7', 'Nome', 'UF',
    'qtd_domicilios_risco', 'pct_domicilios_risco',
    'nota_final', 'capag',
    'total_desastres', 'soma_danos_materiais', 'soma_afetados'
]

# Mantém apenas as colunas que efetivamente existem no seu DataFrame
colunas_presentes = [c for c in colunas_desejadas if c in top20.columns]
top20_export = top20[colunas_presentes].copy()

# Arredondar a nota final para 4 casas decimais
if 'nota_final' in top20_export.columns:
    top20_export['nota_final'] = top20_export['nota_final'].round(4)

# 3. Salvar os arquivos
top20_export.to_csv('top20_municipios_prioritarios.csv', index=False, encoding='utf-8-sig')
top20_export.to_excel('top20_municipios_prioritarios.xlsx', index=False)

print("=== LISTA DOS 20 MUNICÍPIOS CAMPEÕES ===")
if set(['Nome', 'UF', 'qtd_domicilios_risco']).issubset(top20_export.columns):
    print(top20_export[['Cod_IBGE_7', 'Nome', 'UF', 'qtd_domicilios_risco', 'nota_final']].to_string(index=False))

# 4. Disparar o download automático no Colab
files.download('top20_municipios_prioritarios.csv')
files.download('top20_municipios_prioritarios.xlsx')

=== LISTA DOS 20 MUNICÍPIOS CAMPEÕES ===
 Cod_IBGE_7                  Nome UF  qtd_domicilios_risco  nota_final
    4307708                Esteio RS               30000.0      0.9941
    4304606                Canoas RS               70000.0      0.9922
    3304904           São Gonçalo RJ              175838.0      0.9906
    5002704          Campo Grande MS              125000.0      0.9811
    3303708        Paraíba do Sul RJ                8561.0      0.9771
    3301009 Campos dos Goytacazes RJ               63000.0      0.9752
    2601409             Barreiros PE                7938.0      0.9746
    2610004              Palmares PE                8000.0      0.9692
    1600279      Laranjal do Jari AP                6000.0      0.9679
    3113305             Carangola MG                6000.0      0.9655
    3302601           Mangaratiba RJ               11106.0      0.9622
    2913606                Ilhéus BA               20000.0      0.9594
    4216602              São José SC

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 🏁 Conclusão do Estudo de Priorização

Este caderno consolidou o desenvolvimento de uma metodologia transparente, auditável e balanceada para priorização de investimentos e apoio técnico municipal no âmbito do Observatório.

---

#### 📈 Síntese dos Principais Marcos Analíticos

1. **Filtros de Elegibilidade:** Reduziu o universo inicial de **5.570 municípios** para **641 habilitados** usando quatro regras institucionais concomitantes (Participação ativa no SINISA, mapeamento de risco local, exclusão de CAPAGs de alto crédito e presença real de moradias sob perigo climático).
2. **Teste de Sensibilidade:** Demonstrou numericamente o impacto de calibrar o foco estratégico. O cenário recomendado (**70% Quantidade / 30% Percentual de Risco**) provou ser robusto ao equilibrar a escala absoluta das grandes metrópoles com a vulnerabilidade relativa dos pequenos municípios, garantindo a proteção direta de **598.353 moradias** no Top 20.
3. **Custo de Não Agir:** Confirmou a eficiência fiscal do modelo de priorização, demonstrando que o Top 20 sugerido absorve preventivamente uma parcela significativa do passivo de desastres e perdas financeiras históricas.
4. **Arquivos Gerados:** Toda a inteligência geográfica e as posições de ranking foram consolidadas nos arquivos locais `top20_municipios_prioritarios` e `elegiveis_completo_cenarios` para alimentar diretamente painéis BI e mapas no GIS.

In [19]:
import pandas as pd
from google.colab import files

# 1. Garantir os percentis das variáveis
df_elegiveis['rank_qtd_pctile'] = df_elegiveis['qtd_domicilios_risco'].rank(pct=True)
df_elegiveis['rank_pct_pctile'] = df_elegiveis['pct_domicilios_risco'].rank(pct=True)

# 2. Calcular as notas dos 4 cenários
df_elegiveis['nota_100_0'] = (1.0 * df_elegiveis['rank_qtd_pctile']) + (0.0 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_70_30'] = (0.7 * df_elegiveis['rank_qtd_pctile']) + (0.3 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_50_50'] = (0.5 * df_elegiveis['rank_qtd_pctile']) + (0.5 * df_elegiveis['rank_pct_pctile'])
df_elegiveis['nota_0_100'] = (0.0 * df_elegiveis['rank_qtd_pctile']) + (1.0 * df_elegiveis['rank_pct_pctile'])

# 3. Criar a Posição no Ranking (1º, 2º, 3º...) para cada cenário
df_elegiveis['posicao_100_0'] = df_elegiveis['nota_100_0'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_70_30'] = df_elegiveis['nota_70_30'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_50_50'] = df_elegiveis['nota_50_50'].rank(ascending=False, method='min').astype(int)
df_elegiveis['posicao_0_100'] = df_elegiveis['nota_0_100'].rank(ascending=False, method='min').astype(int)

# 4. Criar um marcador simples se o município é Top 20 no cenário recomendado (70/30)
df_elegiveis['is_top20_recomendado'] = df_elegiveis['posicao_70_30'] <= 20

# 5. Ordenar pelo ranking recomendado (70/30)
df_completo_export = df_elegiveis.sort_values(by='posicao_70_30').copy()

# 6. Salvar e realizar o download automático dos 641 elegíveis
df_completo_export.to_csv('elegiveis_completo_cenarios.csv', index=False, encoding='utf-8-sig')
df_completo_export.to_excel('elegiveis_completo_cenarios.xlsx', index=False)

print(f"Sucesso! Exportada a base completa com {len(df_completo_export)} municípios elegíveis e todos os cenários.")

files.download('elegiveis_completo_cenarios.csv')
files.download('elegiveis_completo_cenarios.xlsx')

Sucesso! Exportada a base completa com 641 municípios elegíveis e todos os cenários.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>